<!-- NB_DOC_INTRO_V1 -->
# EDA — Catalogue de visualisations matplotlib / seaborn / plotly

> 📚 **Doc thematique** : [docs/02_EDA.md](docs/02_EDA.md) (EDA & Visualisation)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Catalogue **executable** des types de plots utiles en data science : bar, line (time-series), scatter, boxplot, heatmap, mosaic, parallel coordinates, pie/donut, treemap. Pour chaque : code minimal + cas d'usage + alternatives interactives (plotly).

Pour les stats inferentielles (correlations, tests) : voir [EDA_Stats_Analyse_Desc_Visual.ipynb](./EDA_Stats_Analyse_Desc_Visual.ipynb).

## Plan

1. Bar plot (simple, group, stacked, horizontal)
2. Line / Time-series (avec rolling mean)
3. Scatter + regression
4. Distributions (hist, KDE, boxplot, violin, swarm)
5. Heatmap (correlation, calendar)
6. Pie / donut (et pourquoi a eviter)
7. Mosaic plot (quali × quali)
8. Parallel coordinates (multi-dim)
9. Treemap
10. Plotly interactif
11. Anti-patterns visuels
12. References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Datasets
tips = sns.load_dataset("tips")
flights = sns.load_dataset("flights")
iris = sns.load_dataset("iris")
print(f"tips: {tips.shape}, flights: {flights.shape}, iris: {iris.shape}")

## 1. Bar plot

In [ ]:
# Simple
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Single
agg = tips.groupby("day", observed=True)["total_bill"].mean()
axes[0].bar(agg.index.astype(str), agg.values, color="steelblue")
axes[0].set_title("Simple bar")
axes[0].set_ylabel("total_bill moy")

# 2. Grouped
agg_g = tips.groupby(["day", "sex"], observed=True)["total_bill"].mean().unstack()
agg_g.plot(kind="bar", ax=axes[1])
axes[1].set_title("Grouped bar")
axes[1].tick_params(axis="x", rotation=0)

# 3. Stacked
agg_g.plot(kind="bar", stacked=True, ax=axes[2])
axes[2].set_title("Stacked bar")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 2. Line / Time-series

In [ ]:
# Construire une serie : passengers par annee
ts = flights.groupby("year")["passengers"].sum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ts.index, ts.values, marker="o", label="Passengers")
ax.fill_between(ts.index, ts.values * 0.95, ts.values * 1.05, alpha=0.2, label="±5%")
ax.set_xlabel("Year"); ax.set_ylabel("Passengers")
ax.set_title("Time-series flights")
ax.legend()
plt.show()

# Multiple lines (par categorie)
fig, ax = plt.subplots(figsize=(10, 4))
for month in ["Jan", "Apr", "Jul", "Oct"]:
    sub = flights[flights["month"] == month]
    ax.plot(sub["year"], sub["passengers"], marker="o", label=month)
ax.set_title("Passengers par mois (4 mois)")
ax.legend()
plt.show()

## 3. Scatter + regression

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter simple
axes[0].scatter(tips["total_bill"], tips["tip"], c=tips["sex"].map({"Male": "C0", "Female": "C1"}),
                alpha=0.6, edgecolors="k")
axes[0].set(xlabel="total_bill", ylabel="tip", title="Scatter colore par sex")

# Seaborn regplot (avec regression + CI)
sns.regplot(data=tips, x="total_bill", y="tip", ax=axes[1], scatter_kws={"alpha":0.5})
axes[1].set_title("regplot (regression + 95% CI)")
plt.tight_layout()
plt.show()

## 4. Distributions — hist, KDE, boxplot, violin, swarm

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sns.histplot(tips["total_bill"], kde=True, ax=axes[0, 0])
axes[0, 0].set_title("histplot + KDE")

sns.kdeplot(data=tips, x="total_bill", hue="sex", ax=axes[0, 1], fill=True, alpha=0.5)
axes[0, 1].set_title("KDE par hue")

sns.boxplot(data=tips, x="day", y="total_bill", ax=axes[0, 2])
axes[0, 2].set_title("Boxplot")

sns.violinplot(data=tips, x="day", y="total_bill", ax=axes[1, 0], inner="quartile")
axes[1, 0].set_title("Violin (KDE + quartiles)")

sns.swarmplot(data=tips, x="day", y="total_bill", ax=axes[1, 1], size=3)
axes[1, 1].set_title("Swarm (chaque point)")

sns.stripplot(data=tips, x="day", y="total_bill", ax=axes[1, 2], jitter=0.2, alpha=0.5)
axes[1, 2].set_title("Strip (avec jitter)")

plt.tight_layout()
plt.show()

## 5. Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Correlation
sns.heatmap(iris.select_dtypes(np.number).corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Correlation (iris)")

# Calendar-like : passengers par year × month
ct = flights.pivot(index="month", columns="year", values="passengers")
sns.heatmap(ct, cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Calendar heatmap (flights)")

plt.tight_layout()
plt.show()

## 6. Pie / donut

> ⚠️ **Souvent une mauvaise idee** : difficile a comparer visuellement (les humains comparent mal les angles). Limiter a **<= 5 categories** ou preferer un bar chart.

In [ ]:
agg = tips.groupby("day", observed=True)["total_bill"].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie
axes[0].pie(agg.values, labels=agg.index.astype(str), autopct="%1.1f%%", startangle=90)
axes[0].set_title("Pie (a eviter au-dela de 5 categories)")

# Donut (variante)
wedges, _, _ = axes[1].pie(agg.values, labels=agg.index.astype(str),
                           autopct="%1.1f%%", startangle=90, wedgeprops=dict(width=0.4))
axes[1].set_title("Donut")
plt.tight_layout()
plt.show()

## 7. Mosaic plot — visualiser une table de contingence

In [ ]:
from statsmodels.graphics.mosaicplot import mosaic

fig, ax = plt.subplots(figsize=(8, 5))
mosaic(tips, ["day", "sex"], ax=ax, title="Mosaic plot day × sex (taille = freq)")
plt.tight_layout()
plt.show()

## 8. Parallel coordinates — multi-dim

In [ ]:
from pandas.plotting import parallel_coordinates

fig, ax = plt.subplots(figsize=(10, 5))
parallel_coordinates(iris, "species", ax=ax, colormap=plt.cm.tab10, alpha=0.4)
ax.set_title("Parallel coordinates (iris) — chaque ligne = 1 fleur")
plt.show()

## 9. Treemap (squarify)

In [ ]:
try:
    import squarify
    sizes = tips.groupby("day", observed=True)["total_bill"].sum()
    fig, ax = plt.subplots(figsize=(10, 5))
    squarify.plot(sizes=sizes.values,
                  label=[f"{d}\n${v:.0f}" for d, v in sizes.items()],
                  color=plt.cm.Set3.colors[:len(sizes)],
                  ax=ax, pad=True)
    ax.axis("off")
    ax.set_title("Treemap (total_bill par day)")
    plt.show()
except ImportError:
    print("squarify pas installe : pip install squarify")
    # Fallback : bar chart
    sizes = tips.groupby("day", observed=True)["total_bill"].sum()
    sizes.plot(kind="bar")
    plt.title("Fallback bar (treemap requires `squarify`)")
    plt.show()

## 10. Plotly interactif

In [ ]:
try:
    import plotly.express as px
    fig = px.scatter(tips, x="total_bill", y="tip", color="sex", size="size",
                     facet_col="time", hover_data=["day"],
                     title="Scatter plotly interactif (zoom, hover, etc.)")
    # En notebook : fig.show()
    # En script :
    fig.write_html("/tmp/scatter.html") if False else print("Plotly figure cree (utiliser fig.show() en notebook)")
except ImportError:
    print("plotly pas installe : pip install plotly")

## 11. Anti-patterns visuels

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| **Pie chart > 5 slices** | Bar chart (humain compare mieux les longueurs que les angles) |
| **3D bar / 3D pie** | 2D toujours ; le 3D distord les proportions |
| **Double-axis non-aligne** | Souvent trompeur — preferer 2 plots cote-a-cote |
| **Y axis non a 0** sur bar chart | Exagere les differences |
| **Couleurs arc-en-ciel** (jet, hsv) | Perceptuellement biaisees, illisibles daltoniens — utiliser **viridis** / **cividis** |
| **Pas de label** sur axes / legende | Plot illisible |
| **Sur-densite** de points (overplotting) | `alpha=0.3` ou hex-bin (`plt.hexbin`) ou aggregation |
| **Comparer scales differentes** sans normaliser | Z-score / min-max avant |
| **Pas de titre** | Toujours dire ce que le plot montre |
| **Annotations partout** (chartjunk) | Garder le minimum signifiant |

## 12. References

- **matplotlib** : https://matplotlib.org/stable/
- **seaborn** : https://seaborn.pydata.org/
- **plotly** : https://plotly.com/python/
- **From Data to Viz** (decision tree de viz) : https://www.data-to-viz.com/
- *Storytelling with Data* (Cole Knaflic)
- *Fundamentals of Data Visualization* (Claus Wilke) - gratuit : https://clauswilke.com/dataviz/

### Voir aussi (collection)
- [EDA_Stats_Analyse_Desc_Visual.ipynb](EDA_Stats_Analyse_Desc_Visual.ipynb)
- [EDA_Analyse_Multivarié.ipynb](EDA_Analyse_Multivarié.ipynb)
- [DS_Geospatial.ipynb](DS_Geospatial.ipynb) — cartes
